# Ingesta de productos CSV a Elasticsearch (base de datos vectorial)

**Objetivo del video:** observar cómo pasamos de un archivo CSV a documentos con texto, fragmentos (*chunks*), vectores (*embeddings*) y metadata, y cómo los almacenamos en Elasticsearch para inspeccionarlos en **Elasticvue**.

**Flujo:** `Loader → Chunk → Embedding → Metadata → Storage`.

Este notebook se limita a la **ingesta**. No incluye agentes, recuperación, RAG ni funciones auxiliares.

## Preparación — Instalar dependencias

Usaremos LangChain para cargar y dividir los registros, OpenAI para generar embeddings y el cliente oficial de Elasticsearch para guardar explícitamente texto, vectores y metadata.

In [1]:
%pip install -q langchain-community langchain-text-splitters langchain-openai "elasticsearch>=8,<9" langchain-elasticsearch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Configuración — Completa tus datos

1. Sube el archivo **`productos.csv`** al mismo directorio del notebook (en Colab, `/content/productos.csv`).
2. Completa la clave de OpenAI y la conexión a Elasticsearch con strings.
3. El índice se llamará `productos_vectoriales_demo`. Para ver los datos, ábrelo después desde Elasticvue.

**Nota para el video:** las credenciales que aparecen aquí son marcadores de posición; evita mostrar claves reales durante la grabación.

In [3]:
import os
# Ruta del CSV (en Google Colab, el archivo suele estar en /content/)
CSV_PATH = "/content/productos.csv"

# OpenAI: clave y modelo que convierte texto en vectores
with open("/content/api_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

EMBEDDING_MODEL = "text-embedding-3-small"

# Elasticsearch: dirección del servidor, usuario y contraseña
ES_URL = ""
ES_USER = "elastic"
ES_PASSWORD = ""
INDEX_NAME = ""

## 1. Loader — Leer el CSV

El **loader** convierte **cada fila del CSV** en un objeto `Document` de LangChain. El texto de la fila queda en `page_content` y el origen de la fila aparece en `metadata`.

Para este ejemplo, el CSV contiene las columnas `id`, `nombre`, `categoria`, `marca`, `precio` y `descripcion`. Si usas otro archivo, adapta los nombres de las columnas en la etapa Metadata.

In [4]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path=CSV_PATH, encoding="utf-8")
documentos = loader.load()

print("Productos cargados:", len(documentos))
print("\nTexto del primer producto:\n", documentos[0].page_content)
print("\nMetadata inicial:", documentos[0].metadata)

/tmp/ipykernel_2638/4088280509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


Productos cargados: 10

Texto del primer producto:
 id: P001
nombre: Laptop Ultra 14
categoria: Laptops
marca: NovaTech
precio: 3499
descripcion: Laptop ligera de 14 pulgadas con 16 GB RAM y SSD de 512 GB, ideal para oficina y estudio.

Metadata inicial: {'source': '/content/productos.csv', 'row': 0}


## 2. Chunk — Dividir el contenido

Un **chunk** es un fragmento de texto. El `RecursiveCharacterTextSplitter` separa textos extensos y puede conservar un solapamiento (*overlap*) para no perder contexto entre fragmentos.

Como cada producto del ejemplo es corto, **normalmente quedará en un solo chunk**. Aun así, mostramos este paso para explicar qué ocurriría con descripciones extensas.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Máximo aproximado de caracteres por fragmento
    chunk_overlap=0      # Caracteres compartidos entre fragmentos consecutivos
)

chunks = splitter.split_documents(documentos)

print("Filas originales:", len(documentos))
print("Chunks generados:", len(chunks))
print("\nPrimer chunk:\n", chunks[0].page_content)

Filas originales: 10
Chunks generados: 10

Primer chunk:
 id: P001
nombre: Laptop Ultra 14
categoria: Laptops
marca: NovaTech
precio: 3499
descripcion: Laptop ligera de 14 pulgadas con 16 GB RAM y SSD de 512 GB, ideal para oficina y estudio.


## 3. Embedding — Convertir los chunks en vectores

Un **embedding** representa el significado aproximado del texto mediante una lista de números. Generamos **un vector por chunk** usando el modelo configurado de OpenAI.

Aquí generamos los vectores de forma explícita para poder mostrarlos. En la siguiente etapa los guardaremos tal como se generaron, sin volver a llamar al modelo.

In [6]:
from langchain_openai import OpenAIEmbeddings

modelo_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL
)

vectores = modelo_embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Vectores generados:", len(vectores))
print("Dimensiones de cada vector:", len(vectores[0]))
print("Primeros 8 valores del primer vector:", vectores[0][:8])

Vectores generados: 10
Dimensiones de cada vector: 1536
Primeros 8 valores del primer vector: [-0.0180816650390625, -0.005359649658203125, -0.02093505859375, -0.020477294921875, -0.0189208984375, -0.01071929931640625, -0.009765625, 0.062164306640625]


## 4. Metadata — Agregar información del producto

La **metadata** describe el origen y las características del contenido. A diferencia del embedding, conserva información estructurada para identificar y filtrar productos posteriormente.

Recuperamos `id`, `nombre`, `categoria`, `marca` y `precio` a partir del texto de cada fila leído por el loader. En este ejemplo, esos valores deben coincidir exactamente con los encabezados del CSV. También agregamos el número de chunk.

**Importante:** el texto ya fue convertido en vectores. Esta etapa enriquece los documentos antes de almacenarlos; no altera los embeddings calculados.

In [7]:
import csv

with open(CSV_PATH, encoding="utf-8-sig", newline="") as archivo:
    productos = list(csv.DictReader(archivo))

for numero, chunk in enumerate(chunks):
    # El CSVLoader guarda la posición original en metadata["row"]
    producto = productos[chunk.metadata["row"]]
    chunk.metadata = {
        "producto_id": producto["id"],
        "nombre": producto["nombre"],
        "categoria": producto["categoria"],
        "marca": producto["marca"],
        "precio": float(producto["precio"]),
        "archivo": CSV_PATH,
        "fila_csv": chunk.metadata["row"],
        "chunk_numero": numero
    }

print("Metadata del primer chunk:", chunks[0].metadata)

Metadata del primer chunk: {'producto_id': 'P001', 'nombre': 'Laptop Ultra 14', 'categoria': 'Laptops', 'marca': 'NovaTech', 'precio': 3499.0, 'archivo': '/content/productos.csv', 'fila_csv': 0, 'chunk_numero': 0}


## 5. Storage — Guardar texto, vectores y metadata en Elasticsearch

Creamos un índice de Elasticsearch con tres campos principales:

- `texto`: contenido original del chunk.
- `embedding`: vector numérico en un campo `dense_vector`, para búsquedas por similitud.
- `metadata`: información estructurada del producto.

Usamos el cliente oficial de Elasticsearch para almacenar **los embeddings que ya generamos**. La celda reutiliza un índice existente y sobrescribe los IDs `producto_0`, `producto_1`, etc. si se ejecuta de nuevo con el mismo archivo.

In [9]:
from elasticsearch import Elasticsearch, helpers

# Conexión a Elasticsearch
es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASSWORD)
)

print("Conectado:", es.ping())

# Crear el índice si todavía no existe
if not es.indices.exists(index=INDEX_NAME):
    es.indices.create(
        index=INDEX_NAME,
        mappings={
            "properties": {
                "texto": {"type": "text"},
                "embedding": {
                    "type": "dense_vector",
                    "dims": len(vectores[0]),
                    "index": True,
                    "similarity": "cosine"
                },
                "metadata": {
                    "properties": {
                        "producto_id": {"type": "keyword"},
                        "nombre": {"type": "keyword"},
                        "categoria": {"type": "keyword"},
                        "marca": {"type": "keyword"},
                        "precio": {"type": "float"},
                        "archivo": {"type": "keyword"},
                        "fila_csv": {"type": "integer"},
                        "chunk_numero": {"type": "integer"}
                    }
                }
            }
        }
    )

# Preparar cada fragmento con su vector y metadata
registros = [
    {
        "_index": INDEX_NAME,
        "_id": f"producto_{i}",
        "_source": {
            "metadata": chunk.metadata,
            "texto": chunk.page_content,
            "embedding": vector

        }
    }
    for i, (chunk, vector) in enumerate(zip(chunks, vectores))
]

# Almacenar los documentos en Elasticsearch
insertados, errores = helpers.bulk(
    es,
    registros,
    raise_on_error=False
)

# Actualizar el índice para visualizar los documentos
es.indices.refresh(index=INDEX_NAME)

print("Registros indexados:", insertados)
print("Errores:", len(errores))
print("Índice:", INDEX_NAME)


Conectado: True
Registros indexados: 10
Errores: 0
Índice: productos_vectoriales_demo__b


## Verificación en Elasticvue

Abre **Elasticvue**, conéctate al mismo clúster Elasticsearch y selecciona el índice **`productos_vectoriales_demo`**. Inspecciona los documentos para mostrar estos tres campos:

1. **`texto`**: el producto en lenguaje natural.
2. **`embedding`**: los números que representan su contenido (la interfaz puede resumir el vector).
3. **`metadata`**: ID, categoría, marca, precio y trazabilidad de la fila.

**Mensaje para cerrar el video:** el CSV ya está almacenado en forma de texto, vectores y metadata. La recuperación y los agentes son temas posteriores; en esta práctica solo realizamos la ingesta.

*Si cambias el modelo de embeddings y su dimensión, usa otro nombre de índice o crea uno nuevo con el mapeo correcto.*